In [2]:
#Importing Libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
#Loading Dataset

df = pd.read_csv("WorldBank_CleanedData.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully.
Shape: (2085, 13)


,Region,Country Code,Country,Year,Indicator Code,Indicator Name,Indicator Short Name,Category,Value Type,Sum of Value,Sum of Previous Year Value,Sum of Growth Rate %,Sum of Country Share %
0,East Asia,CHN,China,2000,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,3.478112e-01,NaN,NaN,0.43%
1,East Asia,CHN,China,2000,NE.EXP.GNFS.ZS,Exports of Goods and Services % of GDP,Exports % GDP,Trade,Percentage,2.068160e+01,NaN,NaN,5.52%
2,East Asia,CHN,China,2000,NV.AGR.TOTL.ZS,Agriculture Value Added % of GDP,Agriculture % GDP,Agriculture,Percentage,1.452725e+01,NaN,NaN,7.74%
3,East Asia,CHN,China,2000,NY.GDP.MKTP.CD,GDP Current US$,GDP,Economy,US Dollars,1.223755e+12,NaN,NaN,37.17%
4,East Asia,CHN,China,2000,NY.GDP.PCAP.CD,GDP Per Capita Current US$,GDP per Capita,Economy,US Dollars,9.691995e+02,NaN,NaN,1.45%


In [4]:
#Checking the Columns

print(df.columns.tolist())

['Region', 'Country Code', 'Country', 'Year', 'Indicator Code', 'Indicator Name', 'Indicator Short Name', 'Category', 'Value Type', 'Sum of Value', 'Sum of Previous Year Value', 'Sum of Growth Rate %', 'Sum of Country Share %']


In [5]:
# Make a copy
ml_df = df.copy()

# Rename Power BI exported column names to simpler names
ml_df = ml_df.rename(columns={
    "Sum of Value": "Value",
    "Sum of Previous Year Value": "Previous Year Value",
    "Sum of Growth Rate %": "Growth Rate %",
    "Sum of Country Share %": "Country Share %"
})

# Check new column names
print("Updated columns:")
print(ml_df.columns.tolist())

# Convert important columns to numeric
ml_df["Year"] = pd.to_numeric(ml_df["Year"], errors="coerce")
ml_df["Value"] = pd.to_numeric(ml_df["Value"], errors="coerce")

# Remove rows where Year or Value is missing
ml_df = ml_df.dropna(subset=["Year", "Value"])

# Sort data properly
ml_df = ml_df.sort_values(by=["Country Code", "Indicator Code", "Year"])

print("ML-ready data shape:", ml_df.shape)
ml_df.head()

Updated columns:
['Region', 'Country Code', 'Country', 'Year', 'Indicator Code', 'Indicator Name', 'Indicator Short Name', 'Category', 'Value Type', 'Value', 'Previous Year Value', 'Growth Rate %', 'Country Share %']
ML-ready data shape: (2085, 13)


,Region,Country Code,Country,Year,Indicator Code,Indicator Name,Indicator Short Name,Category,Value Type,Value,Previous Year Value,Growth Rate %,Country Share %
572,Middle East,ARE,United Arab Emirates,2008,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,12.250420,NaN,NaN,9.51%
579,Middle East,ARE,United Arab Emirates,2009,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,1.561813,12.250420,-87.25%,2.43%
586,Middle East,ARE,United Arab Emirates,2010,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,0.877983,1.561813,-43.78%,1.15%
593,Middle East,ARE,United Arab Emirates,2011,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,0.877347,0.877983,-0.07%,1.10%
600,Middle East,ARE,United Arab Emirates,2012,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,0.662269,0.877347,-24.51%,1.01%


In [6]:
# Train Linear Regression and forecast future years

future_years = [2025, 2026, 2027, 2028, 2029]

forecast_rows = []
metrics_rows = []

group_columns = [
    "Country Code",
    "Country",
    "Region",
    "Indicator Code",
    "Indicator Name",
    "Indicator Short Name",
    "Category",
    "Value Type"
]

for group_values, group_data in ml_df.groupby(group_columns):

    group_data = group_data.sort_values("Year")

    # Need at least 5 years of data
    if len(group_data) < 5:
        continue

    X = group_data[["Year"]]
    y = group_data["Value"]

    model = LinearRegression()
    model.fit(X, y)

    # Predict on historical data
    historical_pred = model.predict(X)

    mae = mean_absolute_error(y, historical_pred)
    rmse = np.sqrt(mean_squared_error(y, historical_pred))

    try:
        r2 = r2_score(y, historical_pred)
    except:
        r2 = np.nan

    group_info = dict(zip(group_columns, group_values))

    # Store metrics
    metrics_rows.append({
        **group_info,
        "Model": "Linear Regression",
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    # Store historical actual vs predicted
    for _, row in group_data.iterrows():
        predicted_value = model.predict(pd.DataFrame({"Year": [row["Year"]]}))[0]

        forecast_rows.append({
            **group_info,
            "Year": int(row["Year"]),
            "Actual Value": row["Value"],
            "Predicted Value": predicted_value,
            "Forecast Type": "Historical Prediction",
            "Model": "Linear Regression",
            "MAE": mae,
            "RMSE": rmse,
            "R2 Score": r2
        })

    # Store future forecast rows
    for year in future_years:
        predicted_value = model.predict(pd.DataFrame({"Year": [year]}))[0]

        forecast_rows.append({
            **group_info,
            "Year": year,
            "Actual Value": np.nan,
            "Predicted Value": predicted_value,
            "Forecast Type": "Future Forecast",
            "Model": "Linear Regression",
            "MAE": mae,
            "RMSE": rmse,
            "R2 Score": r2
        })

forecast_df = pd.DataFrame(forecast_rows)
metrics_df = pd.DataFrame(metrics_rows)

print("Forecast dataset created successfully.")
print("Forecast dataset shape:", forecast_df.shape)
forecast_df.head()

Forecast dataset created successfully.
Forecast dataset shape: (2505, 16)


,Country Code,Country,Region,Indicator Code,Indicator Name,Indicator Short Name,Category,Value Type,Year,Actual Value,Predicted Value,Forecast Type,Model,MAE,RMSE,R2 Score
0,ARE,United Arab Emirates,Middle East,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,2008,12.250420,3.674983,Historical Prediction,Linear Regression,2.173754,2.930765,0.101354
1,ARE,United Arab Emirates,Middle East,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,2009,1.561813,3.474072,Historical Prediction,Linear Regression,2.173754,2.930765,0.101354
2,ARE,United Arab Emirates,Middle East,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,2010,0.877983,3.273162,Historical Prediction,Linear Regression,2.173754,2.930765,0.101354
3,ARE,United Arab Emirates,Middle East,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,2011,0.877347,3.072251,Historical Prediction,Linear Regression,2.173754,2.930765,0.101354
4,ARE,United Arab Emirates,Middle East,FP.CPI.TOTL.ZG,Inflation Annual %,Inflation,Economy,Percentage,2012,0.662269,2.871341,Historical Prediction,Linear Regression,2.173754,2.930765,0.101354


In [7]:
#Save forecast files

forecast_df.to_csv("WorldBank_Forecast_Results.csv", index=False)
metrics_df.to_csv("WorldBank_Model_Metrics.csv", index=False)

print("Files saved successfully:")
print("1. WorldBank_Forecast_Results.csv")
print("2. WorldBank_Model_Metrics.csv")

Files saved successfully:
1. WorldBank_Forecast_Results.csv
2. WorldBank_Model_Metrics.csv
